# 🧠 CNN Layer Visualizer — High-Level Animated Edition
Visualizes the full pipeline: **Input → Conv2D → BatchNorm → ReLU → MaxPool → Dropout → Flatten → Dense (Softmax)**

Upgrades over v1:
- Dark theme with design-token color palette
- Progress bar across all scenes
- Architecture overview scene (all layers at once)
- Cell-by-cell animated convolution with real dot-product computation
- Batch Normalization with before/after bar charts + formula
- ReLU with Axes plot + animated matrix transformation
- MaxPool with travelling-dot animation
- Dropout with X-marks on dropped neurons
- Flatten → Dense with probability bars and winner highlight
- Outro summary slide

In [1]:
%%capture
!apt-get install -y -q libpango1.0-dev libcairo2-dev pkg-config python3-dev
!apt-get install -y -q texlive texlive-latex-extra texlive-fonts-extra dvipng
!pip uninstall -y numpy scipy manim
!pip install numpy==1.26.4
!pip install scipy==1.11.4
!pip install manim
!pip install manim-jupyter

### Alternative: Run Manim as a Python script

If the `%%manim` magic command is still not recognized, you can save your Manim scene to a Python file and execute it using the `manim` command-line interface. This achieves the same result.

In [2]:
%%writefile cnnpipeline.py

from manim import *
import numpy as np

# ─────────────────────────────────────────────
#  Design tokens
# ─────────────────────────────────────────────
BG       = "#0D1117"
C_BLUE   = "#58A6FF"
C_GREEN  = "#3FB950"
C_ORANGE = "#F0883E"
C_PURPLE = "#D2A8FF"
C_RED    = "#FF7B72"
C_YELLOW = "#E3B341"
C_TEAL   = "#39D353"
C_GRAY   = "#8B949E"
C_WHITE  = "#E6EDF3"
FONT     = "Monospace"


def section_title(text, color=C_WHITE):
    return Text(text, font=FONT, font_size=34, color=color,
                weight=BOLD).to_edge(UP, buff=0.35)


def subtitle(text, color=C_GRAY, font_size=22):
    return Text(text, font=FONT, font_size=font_size, color=color)


def make_grid(rows, cols, data=None, cell_size=0.55, color=C_BLUE):
    """Build a colored grid with numeric labels."""
    cells = VGroup()
    for r in range(rows):
        for c in range(cols):
            val = data[r][c] if data is not None else 0
            rect = Square(side_length=cell_size,
                          fill_color=color, fill_opacity=0.18,
                          stroke_color=color, stroke_width=1.0)
            num = Text(str(val), font=FONT, font_size=16, color=C_WHITE)
            rect.move_to([c * cell_size, -r * cell_size, 0])
            num.move_to(rect.get_center())
            cells.add(VGroup(rect, num))
    cells.move_to(ORIGIN)
    return cells


# ─────────────────────────────────────────────
#  MAIN SCENE
# ─────────────────────────────────────────────
class CNNPipeline(Scene):

    def setup(self):
        self.camera.background_color = BG

    def progress_bar(self, step, total=8):
        bg = Rectangle(width=12, height=0.16,
                       fill_color=C_GRAY, fill_opacity=0.20,
                       stroke_width=0).to_edge(DOWN, buff=0.18)
        fg = Rectangle(width=12 * step / total, height=0.16,
                       fill_color=C_BLUE, fill_opacity=0.90,
                       stroke_width=0)
        fg.align_to(bg, LEFT).align_to(bg, DOWN)
        return VGroup(bg, fg)

    def clear_all(self):
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)

    # ══════════════════════════════════════════
    def construct(self):
        self.show_intro()
        self.show_architecture_overview()
        self.show_convolution()
        self.show_batch_norm()
        self.show_relu()
        self.show_max_pooling()
        self.show_dropout()
        self.show_flatten_dense()
        self.show_outro()

    # ─── 0  INTRO ────────────────────────────
    def show_intro(self):
        title = Text("CNN Layer Visualizer", font=FONT, font_size=52,
                     color=C_BLUE, weight=BOLD)
        sub = Text("How a Convolutional Neural Network transforms images",
                   font=FONT, font_size=22, color=C_GRAY)
        sub.next_to(title, DOWN, buff=0.4)
        self.play(Write(title), run_time=1.4)
        self.play(FadeIn(sub, shift=UP * 0.15))
        self.wait(1.2)
        self.clear_all()

    # ─── 1  ARCHITECTURE OVERVIEW ────────────
    def show_architecture_overview(self):
        pb = self.progress_bar(1)
        title = section_title("Full CNN Architecture")
        self.play(FadeIn(pb), FadeIn(title, shift=DOWN * 0.2))

        layers = [
            ("Input",     C_WHITE),
            ("Conv2D",    C_BLUE),
            ("BatchNorm", C_PURPLE),
            ("ReLU",      C_GREEN),
            ("MaxPool",   C_ORANGE),
            ("Dropout",   C_RED),
            ("Flatten",   C_YELLOW),
            ("Dense",     C_TEAL),
        ]

        boxes = VGroup()
        for name, color in layers:
            rect = RoundedRectangle(corner_radius=0.15, width=1.5, height=0.55,
                                    fill_color=color, fill_opacity=0.22,
                                    stroke_color=color, stroke_width=2)
            lbl = Text(name, font=FONT, font_size=17, color=color, weight=BOLD)
            lbl.move_to(rect)
            boxes.add(VGroup(rect, lbl))

        boxes.arrange(RIGHT, buff=0.28).scale(0.95).move_to(ORIGIN)

        arrows = VGroup()
        for i in range(len(boxes) - 1):
            a = Arrow(boxes[i].get_right(), boxes[i + 1].get_left(),
                      buff=0.05, color=C_GRAY,
                      stroke_width=2, tip_length=0.15,
                      max_tip_length_to_length_ratio=0.4)
            arrows.add(a)

        self.play(LaggedStart(*[FadeIn(b, shift=UP * 0.15) for b in boxes], lag_ratio=0.12))
        self.play(LaggedStart(*[GrowArrow(a) for a in arrows], lag_ratio=0.1))
        self.play(LaggedStart(
            *[Flash(b, color=b[0].get_stroke_color(),
                    line_length=0.15, flash_radius=0.5) for b in boxes],
            lag_ratio=0.15))
        self.wait(1)
        self.clear_all()

    # ─── 2  CONVOLUTION ──────────────────────
    def show_convolution(self):
        pb = self.progress_bar(2)
        title = section_title("Step 1 — Conv2D: Sliding Kernel", C_BLUE)
        info = subtitle("kernel 3x3  stride 1  padding VALID", C_GRAY)
        info.next_to(title, DOWN, buff=0.18)
        self.play(FadeIn(pb), Write(title), FadeIn(info, shift=UP * 0.1))

        np.random.seed(7)
        raw = np.random.randint(0, 9, (5, 5))
        kernel_vals = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])  # Sobel-X

        input_grid = make_grid(5, 5, raw, color=C_BLUE).scale(0.9).shift(LEFT * 3.8)
        output_grid = make_grid(3, 3, color=C_GREEN).scale(0.9).shift(RIGHT * 3.8)
        kernel_grid = make_grid(3, 3, kernel_vals, cell_size=0.5,
                                color=C_ORANGE).scale(0.85).shift(DOWN * 1.8)
        kernel_box = SurroundingRectangle(kernel_grid, color=C_ORANGE,
                                          corner_radius=0.1, stroke_width=2)
        kernel_lbl = subtitle("Sobel-X Kernel", C_ORANGE, 18)
        kernel_lbl.next_to(kernel_grid, UP, buff=0.18)
        kernel_group = VGroup(kernel_grid, kernel_box, kernel_lbl)

        in_lbl = subtitle("Input (5x5)", C_BLUE, 20).next_to(input_grid, UP, buff=0.2)
        out_lbl = subtitle("Output (3x3)", C_GREEN, 20).next_to(output_grid, UP, buff=0.2)

        self.play(FadeIn(input_grid), FadeIn(in_lbl),
                  FadeIn(output_grid), FadeIn(out_lbl),
                  FadeIn(kernel_group))

        for r in range(3):
            for c in range(3):
                patch = raw[r:r + 3, c:c + 3]
                val = int(np.sum(patch * kernel_vals))

                patch_cells = VGroup(*[input_grid[(r + pr) * 5 + (c + pc)][0]
                                       for pr in range(3) for pc in range(3)])
                highlight = SurroundingRectangle(patch_cells, color=C_ORANGE,
                                                 corner_radius=0.05,
                                                 stroke_width=2.5, buff=0.03)
                out_cell = output_grid[r * 3 + c]
                new_num = Text(str(val), font=FONT, font_size=16, color=C_GREEN)
                new_num.move_to(out_cell[0].get_center())
                flash_rect = out_cell[0].copy().set_fill(C_GREEN, opacity=0.5)

                self.play(Create(highlight), run_time=0.18)
                self.play(Transform(out_cell[1], new_num),
                          FadeIn(flash_rect), run_time=0.22)
                self.play(FadeOut(highlight), FadeOut(flash_rect), run_time=0.12)

        done = subtitle("Convolution complete — edge features extracted",
                        C_GREEN, 20).to_edge(DOWN, buff=0.55)
        self.play(Write(done))
        self.wait(1)
        self.clear_all()

    # ─── 3  BATCH NORMALIZATION ───────────────
    def show_batch_norm(self):
        pb = self.progress_bar(3)
        title = section_title("Step 2 — Batch Normalization", C_PURPLE)
        info = subtitle("Normalises activations so mean=0, std=1 per batch", C_GRAY)
        info.next_to(title, DOWN, buff=0.18)
        self.play(FadeIn(pb), Write(title), FadeIn(info))

        np.random.seed(0)
        n = 12
        raw_vals = np.random.normal(loc=8, scale=4, size=n)
        norm_vals = (raw_vals - raw_vals.mean()) / raw_vals.std()

        def bar_chart(vals, color, label_text, pos):
            bars = VGroup()
            mx = max(abs(vals))
            for i, v in enumerate(vals):
                h = abs(v) / mx * 1.8
                bar = Rectangle(width=0.22, height=max(h, 0.04),
                                fill_color=color, fill_opacity=0.75,
                                stroke_width=0)
                bar.align_to(ORIGIN, DOWN if v >= 0 else UP)
                bar.shift(RIGHT * i * 0.28)
                bars.add(bar)
            bars.move_to(pos)
            axis = Line(bars.get_left() + LEFT * 0.1,
                        bars.get_right() + RIGHT * 0.1,
                        color=C_GRAY, stroke_width=1.5)
            axis.set_y(pos[1])
            lbl = Text(label_text, font=FONT, font_size=20, color=color)
            lbl.next_to(bars, UP, buff=0.25)
            return VGroup(bars, axis, lbl)

        before = bar_chart(raw_vals, C_RED, "Before BN  (mean=8, std=4)", LEFT * 3.2)
        after = bar_chart(norm_vals, C_PURPLE, "After BN  (mean=0, std=1)", RIGHT * 3.2)
        formula = MathTex(r"\hat{x} = \frac{x - \mu}{\sigma}",
                          color=C_PURPLE, font_size=38).shift(DOWN * 2.2)

        self.play(FadeIn(before))
        self.wait(0.4)
        self.play(Write(formula))
        self.wait(0.5)
        arrow = Arrow(before.get_right(), after.get_left(),
                      color=C_PURPLE, stroke_width=2.5, buff=0.25)
        self.play(GrowArrow(arrow))
        self.play(FadeIn(after, shift=RIGHT * 0.2))
        self.play(after[0].animate.set_fill(opacity=1.0), run_time=0.4)
        self.play(after[0].animate.set_fill(opacity=0.75), run_time=0.4)
        self.wait(0.6)
        self.clear_all()

    # ─── 4  ReLU ─────────────────────────────
    def show_relu(self):
        pb = self.progress_bar(4)
        title = section_title("Step 3 — ReLU Activation", C_GREEN)
        info = subtitle("f(x) = max(0, x)  — kills negatives, keeps positives", C_GRAY)
        info.next_to(title, DOWN, buff=0.18)
        self.play(FadeIn(pb), Write(title), FadeIn(info))

        ax = Axes(
            x_range=[-3, 3, 1], y_range=[-0.5, 3, 1],
            x_length=5.5, y_length=3.5,
            axis_config={"color": C_GRAY, "stroke_width": 1.5,
                         "include_tip": True, "tip_length": 0.2},
        ).shift(LEFT * 1.0 + DOWN * 0.4)

        relu_curve = ax.plot(lambda x: max(0.0, x), x_range=[-3, 3],
                             color=C_GREEN, stroke_width=3)
        dead_region = ax.get_area(
            ax.plot(lambda x: 0.0, x_range=[-3, 0]),
            x_range=[-3, 0], color=C_RED, opacity=0.18)
        # Removed 'color' argument as it's no longer supported by get_x_axis_label
        x_lbl = ax.get_x_axis_label("x", direction=RIGHT)
        # Removed 'color' argument as it's no longer supported by get_y_axis_label
        y_lbl = ax.get_y_axis_label("ReLU(x)", direction=UP)

        self.play(Create(ax), Write(x_lbl), Write(y_lbl))
        self.play(FadeIn(dead_region), Create(relu_curve), run_time=1.2)

        dead_lbl = Text("Dead zone (set to 0)", font=FONT, font_size=18, color=C_RED)
        dead_lbl.move_to(ax.c2p(-1.5, 0.5))
        self.play(FadeIn(dead_lbl))

        # Matrix transformation
        data = [[-3, 5, -1], [2, -4, 0], [7, -8, 3]]
        mg = make_grid(3, 3, data, color=C_BLUE).scale(0.9).shift(RIGHT * 4.2)
        mg_lbl = subtitle("Feature map", C_BLUE, 20).next_to(mg, UP, buff=0.2)
        self.play(FadeIn(mg), FadeIn(mg_lbl))

        for i, row in enumerate(data):
            for j, val in enumerate(row):
                if val < 0:
                    cell = mg[i * 3 + j]
                    new_rect = Square(side_length=0.55 * 0.9,
                                      fill_color=C_GREEN, fill_opacity=0.35,
                                      stroke_color=C_GREEN, stroke_width=1.5)
                    new_rect.move_to(cell[0])
                    new_num = Text("0", font=FONT, font_size=16, color=C_GREEN)
                    new_num.move_to(cell[0])
                    self.play(Transform(cell[0], new_rect),
                              Transform(cell[1], new_num), run_time=0.28)

        done = subtitle("All negative activations zeroed out",
                        C_GREEN, 20).to_edge(DOWN, buff=0.55)
        self.play(Write(done))
        self.wait(1)
        self.clear_all()

    # ─── 5  MAX POOLING ──────────────────────
    def show_max_pooling(self):
        pb = self.progress_bar(5)
        title = section_title("Step 4 — MaxPool2D", C_ORANGE)
        info = subtitle("pool 2x2  stride 2  — downsample spatial size by 2x", C_GRAY)
        info.next_to(title, DOWN, buff=0.18)
        self.play(FadeIn(pb), Write(title), FadeIn(info))

        data = [[1, 3, 2, 4], [5, 6, 1, 2], [0, 1, 8, 3], [2, 1, 4, 7]]
        result = [[6, 4], [8, 7]]

        in_g = make_grid(4, 4, data, color=C_BLUE).scale(0.9).shift(LEFT * 3.5)
        out_g = make_grid(2, 2, result, color=C_ORANGE,
                          cell_size=0.75).scale(0.9).shift(RIGHT * 3.5)
        in_l = subtitle("Input (4x4)", C_BLUE, 20).next_to(in_g, UP, buff=0.2)
        out_l = subtitle("Pooled (2x2)", C_ORANGE, 20).next_to(out_g, UP, buff=0.2)
        self.play(FadeIn(in_g), FadeIn(in_l), FadeIn(out_g), FadeIn(out_l))

        # (input_indices, max_flat_index, output_index)
        pools = [
            ([0, 1, 4, 5], 5, 0),
            ([2, 3, 6, 7], 3, 1),
            ([8, 9, 12, 13], 10, 2),
            ([10, 11, 14, 15], 15, 3),
        ]

        for idxs, max_idx, out_idx in pools:
            patch = VGroup(*[in_g[i][0] for i in idxs])
            box = SurroundingRectangle(patch, color=C_ORANGE,
                                       corner_radius=0.05,
                                       stroke_width=2.5, buff=0.03)
            self.play(Create(box), run_time=0.25)
            self.play(Indicate(in_g[max_idx], color=C_ORANGE, scale_factor=1.4), run_time=0.3)

            trav = Dot(in_g[max_idx].get_center(), color=C_ORANGE, radius=0.09)
            out_cell = out_g[out_idx]
            self.play(FadeIn(trav))
            self.play(trav.animate.move_to(out_cell.get_center()),
                      out_cell[0].animate.set_fill(C_ORANGE, opacity=0.6),
                      run_time=0.4)
            self.play(FadeOut(trav), FadeOut(box),
                      out_cell[0].animate.set_fill(C_ORANGE, opacity=0.22),
                      run_time=0.2)

        size_lbl = subtitle("Spatial size halved: 4x4 → 2x2  (75% fewer values)",
                            C_ORANGE, 20).to_edge(DOWN, buff=0.55)
        self.play(Write(size_lbl))
        self.wait(1)
        self.clear_all()

    # ─── 6  DROPOUT ──────────────────────────
    def show_dropout(self):
        pb = self.progress_bar(6)
        title = section_title("Step 5 — Dropout (rate = 0.4)", C_RED)
        info = subtitle("Randomly zero out neurons → prevents overfitting", C_GRAY)
        info.next_to(title, DOWN, buff=0.18)
        self.play(FadeIn(pb), Write(title), FadeIn(info))

        np.random.seed(3)
        n = 16
        neurons = VGroup(*[
            Circle(radius=0.22,
                   fill_color=C_BLUE, fill_opacity=0.8,
                   stroke_color=C_BLUE, stroke_width=1.5)
            for _ in range(n)
        ])
        neurons.arrange_in_grid(rows=2, cols=8, buff=0.4).shift(DOWN * 0.3)
        idx_lbl = subtitle("Active neurons", C_BLUE, 20).next_to(neurons, UP, buff=0.3)
        self.play(FadeIn(neurons), FadeIn(idx_lbl))
        self.wait(0.4)

        drop_mask = np.random.rand(n) < 0.4
        drops = [neurons[i] for i in range(n) if drop_mask[i]]
        keeps = [neurons[i] for i in range(n) if not drop_mask[i]]

        self.play(
            LaggedStart(
                *[neu.animate.set_fill(C_RED, opacity=0.15)
                             .set_stroke(C_RED, opacity=0.4)
                  for neu in drops],
                lag_ratio=0.05),
            run_time=0.8)

        crosses = VGroup(*[Cross(n_obj, stroke_color=C_RED,
                                 stroke_width=2, scale_factor=0.6)
                           for n_obj in drops])
        self.play(LaggedStart(*[FadeIn(x) for x in crosses], lag_ratio=0.04))

        stats = subtitle(
            f"{len(drops)}/{n} neurons dropped  |  {len(keeps)}/{n} kept",
            C_RED, 20).to_edge(DOWN, buff=0.55)
        self.play(Write(stats))
        self.wait(1)
        self.clear_all()

    # ─── 7  FLATTEN + DENSE ──────────────────
    def show_flatten_dense(self):
        pb = self.progress_bar(7)
        title = section_title("Step 6 & 7 — Flatten → Dense (Softmax)", C_YELLOW)
        self.play(FadeIn(pb), Write(title))

        data = [[6, 4], [8, 7]]
        in_g = make_grid(2, 2, data, color=C_ORANGE,
                         cell_size=0.65).scale(0.9).shift(LEFT * 4.0 + DOWN * 0.2)
        in_l = subtitle("Pooled (2x2)", C_ORANGE, 18).next_to(in_g, UP, buff=0.2)
        self.play(FadeIn(in_g), FadeIn(in_l))

        flat_vals = [6, 4, 8, 7]
        flat_cells = VGroup(*[
            VGroup(
                Square(side_length=0.5, fill_color=C_YELLOW, fill_opacity=0.22,
                       stroke_color=C_YELLOW, stroke_width=1.5),
                Text(str(v), font=FONT, font_size=16, color=C_YELLOW)
            )
            for v in flat_vals
        ])
        flat_cells.arrange(DOWN, buff=0.08).shift(LEFT * 0.5 + DOWN * 0.2)
        for cell in flat_cells:
            cell[1].move_to(cell[0])
        flat_lbl = subtitle("Flat (4,)", C_YELLOW, 18).next_to(flat_cells, UP, buff=0.2)

        src_positions = [in_g[i][0].get_center() for i in range(4)]
        for cell, src in zip(flat_cells, src_positions):
            cell.move_to(src)

        self.play(
            LaggedStart(
                *[c.animate.move_to(flat_cells[i][0].get_center())
                  for i, c in enumerate(flat_cells)],
                lag_ratio=0.15),
            run_time=0.9)
        self.play(FadeIn(flat_lbl))

        classes = ["Cat", "Dog", "Bird", "Car"]
        probs = [0.72, 0.18, 0.06, 0.04]
        colors_ = [C_TEAL, C_BLUE, C_PURPLE, C_ORANGE]

        dense_nodes = VGroup(*[
            Circle(radius=0.28, fill_color=c, fill_opacity=0.85,
                   stroke_color=c, stroke_width=1.5)
            for c in colors_],
        )
        dense_nodes.arrange(DOWN, buff=0.32).shift(RIGHT * 2.6 + DOWN * 0.2)

        connectors = VGroup(*[
            Line(fc[0].get_right(), dn.get_left(),
                 stroke_color=C_GRAY, stroke_width=0.5, stroke_opacity=0.35)
            for fc in flat_cells for dn in dense_nodes
        ])
        self.play(FadeIn(connectors))
        self.play(LaggedStart(
            *[FadeIn(n, shift=RIGHT * 0.2) for n in dense_nodes], lag_ratio=0.12))

        bar_group = VGroup()
        for i, (cls, prob, c) in enumerate(zip(classes, probs, colors_)):
            bar = Rectangle(width=prob * 2.8, height=0.32,
                            fill_color=c, fill_opacity=0.85, stroke_width=0)
            bar.next_to(dense_nodes[i], RIGHT, buff=0.25)
            bar.align_to(dense_nodes[i], UP).shift(DOWN * 0.06)
            cls_lbl = Text(f"{cls}  {int(prob * 100)}%",
                           font=FONT, font_size=17, color=c)
            cls_lbl.next_to(bar, RIGHT, buff=0.15)
            bar_group.add(VGroup(bar, cls_lbl))

        self.play(LaggedStart(
            *[GrowFromEdge(bg[0], LEFT) for bg in bar_group], lag_ratio=0.15))
        self.play(LaggedStart(
            *[FadeIn(bg[1]) for bg in bar_group], lag_ratio=0.12))

        winner_box = SurroundingRectangle(
            VGroup(dense_nodes[0], bar_group[0]),
            color=C_TEAL, corner_radius=0.1, stroke_width=2)
        self.play(Create(winner_box))
        self.play(Flash(dense_nodes[0], color=C_TEAL,
                        line_length=0.2, flash_radius=0.5))
        pred = subtitle("Prediction: Cat (72%)", C_TEAL, 26).to_edge(DOWN, buff=0.55)
        self.play(Write(pred))
        self.wait(1.2)
        self.clear_all()

    # ─── 8  OUTRO ────────────────────────────
    def show_outro(self):
        pb = self.progress_bar(8)
        self.play(FadeIn(pb))

        heading = Text("CNN Pipeline Summary", font=FONT, font_size=38,
                       color=C_BLUE, weight=BOLD)
        steps = [
            ("Conv2D",    "Extract spatial features via sliding kernels",  C_BLUE),
            ("BatchNorm", "Stabilise activations  mean=0, std=1",          C_PURPLE),
            ("ReLU",      "Introduce non-linearity — kill negatives",      C_GREEN),
            ("MaxPool",   "Downsample — keep dominant activations",        C_ORANGE),
            ("Dropout",   "Regularise — randomly disable neurons",         C_RED),
            ("Flatten",   "Reshape 2-D maps to 1-D vector",                C_YELLOW),
            ("Dense",     "Map to class probabilities via Softmax",        C_TEAL),
        ]
        rows = VGroup()
        for name, desc, color in steps:
            dot = Dot(color=color, radius=0.1)
            n_t = Text(f"{name}:", font=FONT, font_size=19, color=color, weight=BOLD)
            d_t = Text(desc, font=FONT, font_size=19, color=C_GRAY)
            row = VGroup(dot, n_t, d_t).arrange(RIGHT, buff=0.2)
            rows.add(row)
        rows.arrange(DOWN, buff=0.28, aligned_edge=LEFT)

        group = VGroup(heading, rows).arrange(DOWN, buff=0.45)
        group.move_to(ORIGIN).scale(0.9)

        self.play(Write(heading))
        self.play(LaggedStart(
            *[FadeIn(r, shift=RIGHT * 0.15) for r in rows], lag_ratio=0.1))
        self.wait(2.5)
        self.clear_all()

Overwriting cnnpipeline.py


In [3]:
# Now, run the Manim scene from the created file
# The -qm flags indicate quiet mode and medium quality rendering.
!manim --quality m cnnpipeline.py CNNPipeline

Manim Community v0.20.1

[05/16/26 06:02:55] INFO     Animation 0 : Using cached     ]8;id=506988;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=241806;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             data (hash :                                       
                             1898106350_3486122449_22313245                     
                             7)                                                 
                    INFO     Animation 1 : Using cached     ]8;id=932372;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=467792;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             data (hash :                                       
                             2159456334_98463590_3048755586                

After the Manim rendering is complete, you can find and download the generated video using the following cells:

In [4]:
# Find the rendered video file. Manim typically saves to media/videos/{QUALITY}/{SCENE_NAME}.mp4
import subprocess
import os

# Construct the expected path for the video
video_path = os.path.join(os.getcwd(), 'media', 'videos', '480p15', 'CNNPipeline.mp4')

# Check if the file exists
if os.path.exists(video_path):
    print(f"Video found at: {video_path}")
else:
    print(f"Video not found at the expected path: {video_path}")
    print("Searching for all .mp4 files as a fallback:")
    result = subprocess.run(
        ['find', '/content', '-name', '*.mp4'], # Search within /content only
        capture_output=True, text=True
    )
    print(result.stdout)


Video not found at the expected path: /content/media/videos/480p15/CNNPipeline.mp4
Searching for all .mp4 files as a fallback:
/content/media/videos/cnnpipeline/720p30/CNNPipeline.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_1278090514_1258604283.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_1163420588_2771972669.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_1440743432_2795255541.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_3869732908_1518851225.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_1979828642_1518851225.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_542181513_267707252.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_748689312_2200859980.mp4
/content/media/videos/cnnpipeline/720p30/partial_

In [8]:
from google.colab import files

# Ensure you have the correct path, update if the previous cell shows a different one.
# The video was rendered in 720p30 quality.
video_to_download = '/content/media/videos/cnnpipeline/720p30/CNNPipeline.mp4'

if os.path.exists(video_to_download):
    files.download(video_to_download)
else:
    print(f"Cannot download: Video file not found at {video_to_download}")
    print("Please check the output of the previous cell to confirm the video path.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Find the rendered video file
import subprocess
result = subprocess.run(
    ['find', '/', '-name', '*.mp4', '-not', '-path', '*/proc/*'],
    capture_output=True, text=True
)
print(result.stdout)

/usr/local/lib/python3.12/dist-packages/panel/tests/pane/assets/mp4.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/world.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/b.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/a.mp4
/usr/share/texlive/texmf-dist/tex/latex/mwe/example-movie.mp4
/content/media/videos/cnnpipeline/720p30/CNNPipeline.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_1278090514_1258604283.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_1163420588_2771972669.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_1440743432_2795255541.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_3869732908_1518851225.mp4
/content/media/videos/cnnpipeline/720p30/partial_movie_files/CNNPipeline/2159456334_1979828642_1518851225.mp4
/content/media/videos/cnnpipeline/72

In [9]:
# Download the rendered video (Google Colab only)
from google.colab import files

# The video was rendered in 720p30 quality.
files.download('/content/media/videos/cnnpipeline/720p30/CNNPipeline.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>